In [2]:
# Sección 1: Instalación de dependencias
!pip install --quiet openai langsmith python-dotenv jupyter_bokeh requests pandas openpyxl

# Sección 2: Importación de bibliotecas
import requests
import os
import time
import pandas as pd
from configparser import ConfigParser
from openai import OpenAI

#langsmith
from langsmith import traceable
from langsmith.wrappers import wrap_openai

# Sección 3: Cargar configuración desde archivo
parser = ConfigParser()
parser.read("/content/example.conf")

TOKEN = parser["telegram"]["TOKEN"]
API_KEY = parser["seguridad"]["OPENAI_API_KEY"]

#langsmith
LANGSMITH_API_KEY = parser["seguridad"]["LANGSMITH_API_KEY"]

# Configurar API de Telegram
TELEGRAM_API_URL = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
TELEGRAM_UPDATE_URL = f"https://api.telegram.org/bot{TOKEN}/getUpdates"

#enviroment langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "orderbot-telegram-limar"

# Configurar OpenAI vieja sin langsmith
#client = OpenAI(api_key=API_KEY)
#model = "gpt-4o-mini"


#cliente openai con langsmith
client = wrap_openai(OpenAI(api_key=API_KEY))
model = "gpt-4o-mini"


# Sección 4: Diccionario del menú
MENU = {
    "helado": {
        "1/4 kilo": 2500,
        "1/2 kilo": 4000,
        "1 kilo": 7000,
    },
    "sabores": [
        "Dulce de leche", "Dulce de leche granizado", "Dulce de leche con almendras",
        "Chocolate", "Chocolate amargo", "Chocolate suizo",
        "Banana split", "Frutilla", "Frutilla a la crema",
        "Vainilla", "Limón", "Menta granizada"
    ],
    "toppings": {
        "Rocklets": 500,
        "Kit Kat": 700,
        "Oreo": 350,
    }
}

# Diccionario para almacenar el historial de cada usuario
contextos = {}

def get_user_context(chat_id):
    """Obtiene el historial de conversación de un usuario o lo inicializa si no existe."""
    if chat_id not in contextos:
        contextos[chat_id] = [{
            'role': 'system',
            'content': f"""
Eres OrderBot, un servicio automatizado para recolectar pedidos de Helados,
una heladería local llamada LIMAR ubicada en Av La Plata al 1500. Primero saludas al cliente, y muestra el menú y los precios.
luego recolectas el pedido teniendo en cuenta no ofrecer ni vender nada que no este específicado en el contexto. Recuerda no limitar los pedidos del
cliente en cuanto a cantidades y formatos del pedido, por ejemplo no puede pedir más de 1 kilo por gusto, aqui no hay limites.
Esperas para recolectar todo el pedido, permitiendo que agregue más helado o cualquier otra cosa del menú las veces que sea necesario.
Finalmente, cuando ya recolectaste el pedido, confirmas con el usuario la lista completa del pedido con su cuenta total asegurándote de hacer las cuentas
correctamente es decir si son 2 kilos hacer el producto 2 por el precio de un kilo, y pides que si hay errores te informen.
Luego preguntas si pagará en efectivo o tarjeta y finalmente preguntas si es para recoger o para entregar. Si es una entrega, pides dirección.
Finalmente, escribes literalmente: 'Los empleados le confirmarán el total (por si haya un error en mis cálculos) y le cobrarán.'

Menú:
- Cuarto kilo de helado: ${MENU['helado']['1/4 kilo']}
- Medio kilo de helado: ${MENU['helado']['1/2 kilo']}
- Un kilo de helado: ${MENU['helado']['1 kilo']}

Sabores: {", ".join(MENU['sabores'])}

Toppings:
{chr(10).join([f"- {t}: ${MENU['toppings'][t]}" for t in MENU['toppings']])}
"""
        }]
    return contextos[chat_id]

# Función para enviar mensajes a Telegram con langsmith
@traceable(name="Enviar mensaje a Telegram")
def send_message(text, chat_id):
    try:
        response = requests.post(
            TELEGRAM_API_URL,
            json={"chat_id": chat_id, "text": text}
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al enviar mensaje: {e}")
        raise

# Nueva Función para procesar mensajes del usuario con langsmith
@traceable(name="Procesar mensaje del usuario")
def process_message(user_message, chat_id):
    context = get_user_context(chat_id)
    context.append({'role': 'user', 'content': user_message})

    response = client.chat.completions.create(
        model=model,
        messages=context
    )

    bot_reply = response.choices[0].message.content

    send_message(bot_reply, chat_id)

    context.append({'role': 'assistant', 'content': bot_reply})

    if "Los empleados le confirmarán el total" in bot_reply:
        tickets_pedidos(chat_id, bot_reply)



# Función para registrar pedidos en un archivo Excel con langsmith
@traceable(name="Guardar pedido en Excel")
def tickets_pedidos(chat_id, content):
    archivo_excel = "pedidos.xlsx"

    if os.path.exists(archivo_excel):
       df = pd.read_excel(archivo_excel, engine="openpyxl")
    else:
       df = pd.DataFrame(columns=["Chat_ID", "Mensaje"])

    nueva_fila = pd.DataFrame([{"Chat_ID": chat_id, "Mensaje": content}])
    df = pd.concat([df, nueva_fila], ignore_index=True)
    df.to_excel(archivo_excel, index=False, engine="openpyxl")


# Función para obtener actualizaciones desde Telegram con langsmith
@traceable(name="Obtener actualizaciones de Telegram")
def get_updates(offset=None):
    params = {"timeout": 30, "offset": offset}
    try:
        response = requests.get(TELEGRAM_UPDATE_URL, params=params)
        response.raise_for_status()
        data = response.json()
        return data.get("result", [])
    except requests.exceptions.RequestException as e:
        print(f"Error al obtener actualizaciones: {e}")
        raise

# Bucle principal del chatbot
last_update_id = None
while True:
    updates = get_updates(last_update_id)
    for update in updates:
        if "message" in update:
            chat_id = update["message"]["chat"]["id"]
            user_message = update["message"].get("text", "")
            if user_message:
                process_message(user_message, chat_id)
            last_update_id = update["update_id"] + 1
    time.sleep(2)


ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import os

print(os.environ.get("LANGSMITH_TRACING"))
print(os.environ.get("LANGSMITH_PROJECT"))
print(os.environ.get("LANGSMITH_API_KEY")[:8])

true
orderbot-telegram-limar
lsv2_pt_


In [ ]:
from langsmith import traceable

@traceable(name="prueba_manual_langsmith", project_name="orderbot-telegram-limar")
def prueba_langsmith():
    return {"resultado": "LangSmith funciona"}

print(prueba_langsmith())

{'resultado': 'LangSmith funciona'}
